In [ ]:
# ==== 0. Imports & Global Config =====================================
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import Dataset, DataLoader

import shap
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_context("talk")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sleep-edf-database-expanded-1.0.0").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ROOT = PROJECT_ROOT / "sleep-edf-database-expanded-1.0.0"
NUM_CLASSES = 5
CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# ==== 1. Phase 1: Epoching PSG+Hypnogram  ===========================
import mne 

PAIRS = [...] 

OUT = ROOT / "derived_sc"
OUT.mkdir(exist_ok=True)

def epoch_one(psg_path, hyp_path, epoch_len=30.0,
              pick=("EEG Fpz-Cz","EEG Pz-Oz","EOG horizontal","EMG submental")):
    # Load PSG
    raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
    raw.pick([ch for ch in pick if ch in raw.ch_names])

    # Load hypnogram and convert to a single Annotations object with mapped labels
    hyp = mne.read_annotations(hyp_path)  # no verbose kw on older MNE
    onsets, durations, desc = [], [], []
    for i in range(len(hyp)):
        d = hyp.description[i]
        if d in KEEP:
            onsets.append(hyp.onset[i])
            durations.append(hyp.duration[i])
            desc.append(LABEL_MAP[d])

    if not onsets:   # no usable labels
        return None, None

    ann = mne.Annotations(onset=onsets,
                          duration=durations,
                          description=desc,
                          orig_time=getattr(hyp, "orig_time", None))
    raw.set_annotations(ann)

    # Events from annotations using our fixed event id
    events, _ = mne.events_from_annotations(raw, event_id=CLASS_TO_ID, verbose=False)

    # Epoch into 30 s segments (slightly <30s to avoid right-edge issues)
    epochs = mne.Epochs(raw, events, CLASS_TO_ID,
                        tmin=0, tmax=30 - 1/raw.info["sfreq"],
                        baseline=None, preload=True,
                        reject_by_annotation=True, verbose=False)

    X = epochs.get_data()       # (n_epochs, n_channels, n_samples)
    y = epochs.events[:, -1]    # integers 0..4
    return X, y

def save_npz(psg, hyp):
    rec_id = psg.stem.split("-")[0]
    X, y = epoch_one(psg, hyp, ...)
    if X is None:
        print("Skipped:", rec_id)
        return 0
    np.savez_compressed(OUT / f"{rec_id}.npz", X=X, y=y)
    print(rec_id, X.shape[0], "epochs")
    return X.shape[0]

total_epochs = sum(save_npz(psg, hyp) for psg, hyp in PAIRS)
print("Total epochs saved:", total_epochs)
